In [23]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from numb_project.domain_module import *
from numb_project.gw_model import *
from numb_project.data_module import *
from cfg_tools import load_config_files

end_chain = 3

@torch.no_grad()
@torch.no_grad()
def evaluate_chain_accuracy(
    model: MyGlobalWorkspace,
    dataloader,
    base: int = BASE,
    chain_length: int = end_chain * 2,
    start_chain: int = 0,
    end_chain: int = 10,
    n_batches: int | None = None,
    device: str = "cuda",
):
    model.eval()
    model.to(device)

    correct_at_target_step = 0
    total = 0
    step_correct = torch.zeros(chain_length, device=device)
    step_total = torch.zeros(chain_length, device=device)

    group_key = frozenset({"image", "digit"})

    for batch_idx, item in enumerate(dataloader):
        if n_batches is not None and batch_idx >= n_batches:
            break

        # CombinedLoader yields (batch_dict, batch_idx, dataloader_idx)
        batch_dict = item[0] if isinstance(item, tuple) else item
        group = batch_dict[group_key]

        image = group["image"].to(device)
        left_digit_one_hot = group["digit"].to(device)

        right_addend_onehot, target_one_hot = construct_add_chain_task(
            left_digit_one_hot, base=base, start_chain=start_chain, end_chain=end_chain
        )
        right_addend_onehot = right_addend_onehot.to(device)
        target_idx = target_one_hot.argmax(dim=1).to(device)
        right_addend_value = right_addend_onehot.argmax(dim=1)

        cumulative_preds = model.forward_chain(
            image, right_addend_onehot, left_digit_one_hot, chain_length=chain_length
        )

        batch_size = image.shape[0]

        preds_at_target_step = cumulative_preds[
            torch.arange(batch_size, device=device), right_addend_value, :
        ]
        pred_idx = preds_at_target_step.argmax(dim=1)
        correct_at_target_step += (pred_idx == target_idx).sum().item()
        total += batch_size

        left_idx = left_digit_one_hot.argmax(dim=1)
        for t in range(chain_length):
            expected_t = (left_idx + t) % base
            pred_t = cumulative_preds[:, t, :].argmax(dim=1)
            step_correct[t] += (pred_t == expected_t).sum()
            step_total[t] += batch_size

    target_step_acc = correct_at_target_step / total
    per_step_acc = (step_correct / step_total).cpu()

    print(f"Accuracy au pas cible exact : {target_step_acc:.4f} ({correct_at_target_step}/{total})")
    print("\nAccuracy par pas t :")
    for t, acc in enumerate(per_step_acc):
        print(f"  t={t:2d} : {acc:.4f}")

    return target_step_acc, per_step_acc

@torch.no_grad()
def show_qualitative_examples(
    model: MyGlobalWorkspace,
    dataloader,
    base: int = BASE,
    chain_length: int = BASE * 2,
    n_examples: int = 5,
    device: str = "cuda",
):
    model.eval()
    model.to(device)

    group_key = frozenset({"image", "digit"})

    item = next(iter(dataloader))
    batch_dict = item[0] if isinstance(item, tuple) else item
    group = batch_dict[group_key]

    image = group["image"][:n_examples].to(device)
    left_digit_one_hot = group["digit"][:n_examples].to(device)

    right_addend_onehot, target_one_hot = construct_add_chain_task(left_digit_one_hot, base=base)
    right_addend_onehot = right_addend_onehot.to(device)
    target_idx = target_one_hot.argmax(dim=1).to(device)
    right_addend_value = right_addend_onehot.argmax(dim=1)

    cumulative_preds = model.forward_chain(
        image, right_addend_onehot, left_digit_one_hot, chain_length=chain_length
    )
    left_idx = left_digit_one_hot.argmax(dim=1)

    for i in range(n_examples):
        steps_needed = right_addend_value[i].item()
        pred_seq = cumulative_preds[i, :, :].argmax(dim=1).cpu().tolist()
        pred_at_target = pred_seq[steps_needed]
        print(
            f"Exemple {i}: gauche={left_idx[i].item()}, droite(+{steps_needed}), "
            f"cible={target_idx[i].item()}, prédiction au pas {steps_needed}={pred_at_target}, "
            f"{'OK' if pred_at_target == target_idx[i].item() else 'FAIL'}"
        )
        print(f"  séquence complète des prédictions (t=0..{chain_length-1}): {pred_seq}")

config = load_config_files(
    f"/home/lucas/gwnumb/config",
    use_cli=False,
    load_files=["config.yaml"])[0]

domain_configs = get_domains_config(['image', 'digit'])

# --- usage ---
data_module = MnistDataModule(config["training"]["batch_size"])
test_loader = data_module.test_dataloader()
gw_mod, selection_mod, operation_mod, attention_mod, loss_mod = get_global_workspace_mods(config, domain_configs)

model = MyGlobalWorkspace.load_from_checkpoint(
    "/home/lucas/gwnumb/checkpoints/numb/debug.ckpt",
    gw_mod=gw_mod, selection_mod=selection_mod, loss_mod=loss_mod,
    operation_mod=operation_mod, attention_mod=attention_mod,
    weights_only=False
)

evaluate_chain_accuracy(model, test_loader, device="cuda", end_chain=end_chain)
show_qualitative_examples(model, test_loader, device="cuda")

Accuracy au pas cible exact : 0.1016 (1016/10000)

Accuracy par pas t :
  t= 0 : 0.0980
  t= 1 : 0.0714
  t= 2 : 0.0711
  t= 3 : 0.1010
  t= 4 : 0.1032
  t= 5 : 0.1135
Exemple 0: gauche=3, droite(+2), cible=5, prédiction au pas 2=6, FAIL
  séquence complète des prédictions (t=0..19): [0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]
Exemple 1: gauche=7, droite(+0), cible=7, prédiction au pas 0=0, FAIL
  séquence complète des prédictions (t=0..19): [0, 7, 7, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]
Exemple 2: gauche=2, droite(+8), cible=0, prédiction au pas 8=6, FAIL
  séquence complète des prédictions (t=0..19): [0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]
Exemple 3: gauche=8, droite(+3), cible=1, prédiction au pas 3=6, FAIL
  séquence complète des prédictions (t=0..19): [0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]
Exemple 4: gauche=2, droite(+7), cible=9, prédiction au pas 7=6, FAIL
  séquence complète des prédictions (t=0..19): [0,